# Qiskit Port Tutorial: AIM Workflow Validation

This notebook is a tutorial-style validation notebook for the Qiskit port of the Anderson Impurity Model (AIM) repository.

The goal is to mirror the spirit of `aim_tutorial.ipynb`, but focus only on the Qiskit-port components that have been validated so far.

This notebook does **not** yet run the full Qiskit VQE/Lanczos workflow. Instead, it verifies the pieces needed before full integration:

1. Initialize a real AIM system from the repository.
2. Compare Qulacs and Qiskit/NumPy helper operations.
3. Validate statevector ordering.
4. Validate Hamiltonian matrix construction.
5. Validate continued-fraction recursion.
6. Document current limitations and next steps.

## 0. Setup

Before running this notebook, clone the repository and create the Python environment.

```bash
git clone <repository-url>
cd qcaob-aim
```

Create and activate a virtual environment.

For macOS/Linux:

```bash
python3 -m venv qiskit-clean
source qiskit-clean/bin/activate
```

For Windows:

```bash
python -m venv qiskit-clean
qiskit-clean\Scripts\activate
```

Install the project dependencies.

```bash
pip install -r requirements.txt
```

If Qiskit packages are not included in the requirements file, install them separately.

```bash
pip install qiskit qiskit-aer
```

---

# Launch Jupyter

From the repository root, launch Jupyter.

```bash
jupyter lab or jupyter notebook
```

Then open:

```
qiskit_port_tutorial.ipynb
```

---

# Repository Layout

The tutorial uses both the original repository implementation and the Qiskit port.

```
qcaob-aim/
│
├── dmft.py
├── vqe.py
├── anderson_impurity_model.py
│
├── qiskit_port/
│   ├── qiskit_vqe.py
│   ├── qiskit_ansatz.py
│   └── ...
│
└── qiskit_port_tutorial.ipynb
```



## 1. Imports

This section imports the original repository modules and the new Qiskit-port modules.

The original repository is used as the reference implementation.  
The Qiskit port is tested against it function by function.

In [1]:
import numpy as np
import importlib

import dmft
import vqe
import anderson_impurity_model as aim

import qiskit_port.qiskit_vqe as qv
import qiskit_port.qiskit_ansatz as qa

from qulacs import QuantumState
from qulacs.state import inner_product
from qulacs.quantum_operator import create_quantum_operator_from_openfermion_text
from qiskit.quantum_info import Statevector

importlib.reload(qv)
importlib.reload(qa)

print("Imports complete.")

Imports complete.


## 2. Initialize a Real AIM System

This uses the original repository function `dmft.initialize_system()`.

This gives real AIM objects:

- impurity orbital
- AIM model
- spin-up and spin-down qubit registers
- connected graphs
- OpenFermion qubit Hamiltonian

This does not run VQE yet.

In [12]:
system_size = 3
seed = 0

(
    impurity_orbital,
    test_model,
    n_orbitals,
    up_qubit_indices,
    down_qubit_indices,
    connected_graphs,
    qubit_hamiltonian,
) = dmft.initialize_system(system_size, seed)

print("System size:", system_size)
print("Seed:", seed)
print("n_orbitals:", n_orbitals)
print("up_qubit_indices:", up_qubit_indices)
print("down_qubit_indices:", down_qubit_indices)
print("connected graph keys:", connected_graphs.keys())
print("qubit Hamiltonian:")
print(qubit_hamiltonian)

System size: 3
Seed: 0
n_orbitals: 3
up_qubit_indices: [0, 1, 2]
down_qubit_indices: [3, 4, 5]
connected graph keys: dict_keys(['graph_full', 'graph_up', 'graph_down', 'graph_stitch'])
qubit Hamiltonian:
(3.891581783524612+0j) [] +
(-0.397142095845775+0j) [X0 X1] +
(-0.397142095845775+0j) [Y0 Y1] +
(-0.05637360684304271+0j) [Z0] +
(-1.2054162485351831+0j) [X1 X2] +
(-1.2054162485351831+0j) [Y1 Y2] +
(-3.439721180632871+0j) [Z1] +
(2.1499491659313583+0j) [Z1 Z4] +
(0.47532931274792833+0j) [Z2] +
(-0.397142095845775+0j) [X3 X4] +
(-0.397142095845775+0j) [Y3 Y4] +
(-0.05637360684304271+0j) [Z3] +
(-1.2054162485351831+0j) [X4 X5] +
(-1.2054162485351831+0j) [Y4 Y5] +
(-3.439721180632871+0j) [Z4] +
(0.47532931274792833+0j) [Z5]


## 3. Helper Function Validation

The first Qiskit-port helper functions replace basic Qulacs operations.

| Original Qulacs operation | Qiskit/NumPy replacement |
|---|---|
| `get_expectation_value()` | `expectation_value()` |
| `inner_product()` | `overlap()` |
| `get_transition_amplitude()` | `transition_amplitude()` |

These are controlled unit tests: the same Hamiltonian/state inputs are given to both the original Qulacs operation and the Qiskit-port helper function.

In [16]:
from qiskit_port.qiskit_vqe import expectation_value, transition_amplitude, overlap

# H = Z0
qulacs_H = create_quantum_operator_from_openfermion_text("1.0 [Z0]")
Hmat = np.array([[1, 0], [0, -1]], dtype=complex)

for basis_state in [0, 1]:
    qs = QuantumState(1)
    qs.set_computational_basis(basis_state)

    state = np.zeros(2, dtype=complex)
    state[basis_state] = 1.0

    qulacs_result = qulacs_H.get_expectation_value(qs)
    qiskit_result = expectation_value(state, Hmat)

    print(f"|{basis_state}>")
    print("Qulacs:", qulacs_result)
    print("Qiskit:", qiskit_result)
    print("Difference:", abs(qulacs_result - qiskit_result))
    print("-" * 40)

    assert abs(qulacs_result - qiskit_result) < 1e-12

print("expectation_value() matches Qulacs.")

|0>
Qulacs: (1+0j)
Qiskit: (1+0j)
Difference: 0.0
----------------------------------------
|1>
Qulacs: (-1+0j)
Qiskit: (-1+0j)
Difference: 0.0
----------------------------------------
expectation_value() matches Qulacs.


## 4. Statevector Ordering Validation

The original repository uses `vqe.qulacs_to_python_ordering()` to convert Qulacs statevector ordering into Python/Qiskit-style ordering.

The Qiskit port implements the equivalent function:

`qiskit_port.qiskit_vqe.qulacs_to_python_ordering_qiskit()`

This test compares the two directly.

In [6]:
n_qubits = 3
state = np.arange(2**n_qubits, dtype=complex)

original = vqe.qulacs_to_python_ordering(state, n_qubits)
converted = qv.qulacs_to_python_ordering_qiskit(state, n_qubits)

print("Input:     ", state)
print("Original:  ", original)
print("Converted: ", converted)
print("Max difference:", np.max(np.abs(original - converted)))

assert np.allclose(original, converted)
print("qulacs_to_python_ordering_qiskit() matches original.")

Input:      [0.+0.j 1.+0.j 2.+0.j 3.+0.j 4.+0.j 5.+0.j 6.+0.j 7.+0.j]
Original:   [0.+0.j 4.+0.j 2.+0.j 6.+0.j 1.+0.j 5.+0.j 3.+0.j 7.+0.j]
Converted:  [0.+0.j 4.+0.j 2.+0.j 6.+0.j 1.+0.j 5.+0.j 3.+0.j 7.+0.j]
Max difference: 0.0
qulacs_to_python_ordering_qiskit() matches original.


## 5. Hamiltonian Matrix Validation

The original repo constructs:

- a shifted Qulacs Hamiltonian
- a squared shifted Qulacs Hamiltonian
- the number of qubits

using:

`vqe.create_qulacs_hamiltonian()`

The Qiskit port constructs equivalent dense matrices using OpenFermion:

`qiskit_port.qiskit_vqe.create_qiskit_hamiltonian_matrix()`

Validation is performed by comparing expectation values of \(H\) and \(H^2\) on every computational basis state after applying the Qulacs-to-Qiskit ordering conversion.

In [7]:
# Temporary shift value for validating the construction only
vqe_gs_energy = -1.0

qulacs_H, qulacs_H2, n_qubits_qulacs = vqe.create_qulacs_hamiltonian(
    qubit_hamiltonian,
    vqe_gs_energy
)

Hmat, H2mat, n_qubits_qiskit = qv.create_qiskit_hamiltonian_matrix(
    qubit_hamiltonian,
    vqe_gs_energy
)

print("Qulacs n_qubits:", n_qubits_qulacs)
print("Qiskit n_qubits:", n_qubits_qiskit)

assert n_qubits_qulacs == n_qubits_qiskit

Qulacs n_qubits: 4
Qiskit n_qubits: 4


In [8]:
from qiskit_port.qiskit_vqe import qulacs_to_python_ordering_qiskit

max_diff_H = 0.0
max_diff_H2 = 0.0

max_qulacs_exp_H = -np.inf
max_qiskit_exp_H = -np.inf

max_qulacs_exp_H2 = -np.inf
max_qiskit_exp_H2 = -np.inf

for basis in range(2**n_qubits_qulacs):
    qs = QuantumState(n_qubits_qulacs)
    qs.set_computational_basis(basis)

    qulacs_state = np.zeros(2**n_qubits_qulacs, dtype=complex)
    qulacs_state[basis] = 1.0

    state = qulacs_to_python_ordering_qiskit(qulacs_state, n_qubits_qulacs)

    qulacs_exp_H = qulacs_H.get_expectation_value(qs).real
    qiskit_exp_H = np.vdot(state, Hmat @ state).real

    qulacs_exp_H2 = qulacs_H2.get_expectation_value(qs).real
    qiskit_exp_H2 = np.vdot(state, H2mat @ state).real

    max_qulacs_exp_H = max(max_qulacs_exp_H, qulacs_exp_H)
    max_qiskit_exp_H = max(max_qiskit_exp_H, qiskit_exp_H)

    max_qulacs_exp_H2 = max(max_qulacs_exp_H2, qulacs_exp_H2)
    max_qiskit_exp_H2 = max(max_qiskit_exp_H2, qiskit_exp_H2)

    max_diff_H = max(max_diff_H, abs(qulacs_exp_H - qiskit_exp_H))
    max_diff_H2 = max(max_diff_H2, abs(qulacs_exp_H2 - qiskit_exp_H2))

print("===== H =====")
print("Max Qulacs expectation :", max_qulacs_exp_H)
print("Max Qiskit expectation :", max_qiskit_exp_H)
print("Max difference         :", max_diff_H)

print("\n===== H^2 =====")
print("Max Qulacs expectation :", max_qulacs_exp_H2)
print("Max Qiskit expectation :", max_qiskit_exp_H2)
print("Max difference         :", max_diff_H2)

assert max_diff_H < 1e-10
assert max_diff_H2 < 1e-10

print("\nHamiltonian matrix matches Qulacs after ordering conversion.")

===== H =====
Max Qulacs expectation : 14.758884722531484
Max Qiskit expectation : 14.758884722531484
Max difference         : 4.440892098500626e-16

===== H^2 =====
Max Qulacs expectation : 219.08645300731536
Max Qiskit expectation : 219.08645300731538
Max difference         : 2.842170943040401e-14

Hamiltonian matrix matches Qulacs after ordering conversion.


## 6. Continued Fraction Validation

The original repository computes Green's functions from Lanczos coefficients using:

`vqe.continued_fraction()`

The Qiskit port implements:

`qiskit_port.qiskit_vqe.continued_fraction_qiskit()`

This test compares the recursive continued-fraction implementation directly.

This does **not** yet validate generation of Lanczos coefficients. It only validates that the continued-fraction recursion itself has been ported correctly.

In [9]:
w = np.linspace(-5, 5, 20, dtype=np.complex128) + 0.1j
a = np.array([1.0, 2.0, 3.0], dtype=np.complex128)
b = np.array([0.0, 0.5, 0.25], dtype=np.complex128)

original = vqe.continued_fraction(w, a, b)
converted = qv.continued_fraction_qiskit(w, a, b)

diff = np.max(np.abs(original - converted))

print("Max difference:", diff)

assert np.allclose(original, converted)
print("continued_fraction_qiskit() matches original vqe.continued_fraction().")

Max difference: 0.0
continued_fraction_qiskit() matches original vqe.continued_fraction().


## 7. Generic Ansatz Status

A first Qiskit version of the generic symmetry-preserving ansatz has been implemented in:

`qiskit_port/qiskit_ansatz.py`

This section is intentionally marked as **in progress**.

The current Qiskit ansatz is being debugged against the original Qulacs gate sequence. The state overlap is high, but the statevectors do not yet match exactly, which indicates a likely gate convention, ordering, or rotation-sign mismatch.

This is the next major validation target before `construct_vqe_gs()` and the variational Lanczos routines can be fully ported.

In [10]:
print("Current ansatz validation status:")
print("- Qiskit generic ansatz scaffold exists.")
print("- Direct statevector comparison is still being debugged.")
print("- Full VQE/Lanczos integration is pending ansatz validation.")

Current ansatz validation status:
- Qiskit generic ansatz scaffold exists.
- Direct statevector comparison is still being debugged.
- Full VQE/Lanczos integration is pending ansatz validation.


## 8. Current Status

Validated:

- helper functions
- statevector ordering conversion
- Hamiltonian matrix construction
- continued-fraction recursion

In progress:

- generic symmetry-preserving ansatz

Pending:

- `construct_vqe_gs()`
- `lanczos_cost_function()`
- `vqe_ideal_lanczos_iterations()`
- `vqe_gf_pm()`
- `calculate_gf_vqe()`
- `dmft.py` backend integration

The full Qiskit VQE/Lanczos workflow is not complete yet. This notebook provides a reproducible tutorial-style validation record of the completed conversion pieces.